[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Dpkt](Dpkt.md) | [Pyarrow](Pyarrow.md) | [Tasks](Tasks.md) | [Task 7 Guide](Task-hilbert-map.md) | [flow compare](flow_compare.ipynb) | flow analysis | [Slides](slides/)

### Network Infrastructure Data Science (NIDS) Module
---
# Investigating traffic surge in network telescope

In this part, we will employ flow analysis to identify the root cause(s) of traffic surge.

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
%pip install -q pandas pyarrow pybgpkit-parser pelicanfs pytricia geoip2 maxminddb matplotlib

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import struct
import pickle
import contextlib
import gzip
import io
import urllib.parse
import urllib.request
from collections import defaultdict

import pybgpkit_parser as bgpkit
from pelicanfs.core import OSDFFileSystem
import pytricia

import geoip2.database
import maxminddb

import matplotlib.pyplot as plt

In [ ]:
# helper for reading Maxmind DB from Ceph S3 endpoint
def _open_binary_stream(source):
    """Open source for reading raw bytes.

    source may be a local file path or an http(s) URL (e.g. a presigned
    S3 link). Either way this returns an unbuffered-content stream
    supporting .read(); it is not decompressed yet.
    """
    parsed = urllib.parse.urlparse(source)
    if parsed.scheme in ("http", "https"):
        return urllib.request.urlopen(source)
    return open(source, "rb")

In [ ]:
# Names for the parquet files to be read back in. These MUST match the names
# flow_compare.ipynb wrote - if you changed them there, change them here too.
FLOWS_A_OUT = "ucsd-nt-sub.1782463980.flow.parquet"
FLOWS_B_OUT = "ucsd-nt-sub.1782464400.flow.parquet"

# names for the MaxMind DB and RouteView collector to be used for geolocation and ASN lookups.
GEOIP_DB = "http://rook-ceph-rgw-nautiluss3.rook/caida/geolocation/maxmind/2026-06-24.GeoLite2-City.mmdb.gz"
COLLECTOR = "route-views3"
OSDF_BASE = "https://osdf-director.osg-htc.org"
ASN_CACHE = "prefix_to_asn.pkl"

# Month of the RouteViews RIB archive used to resolve origin ASes. We take the
# first RIB of this month, while the captures are from 2026-06-26, so the
# routing view is a few weeks older than the traffic. Prefix origins are stable
# enough over that window for this assignment, but it is an approximation.
RIB_MONTH = "2026.06"

# Read back in the two flow parquet files you created

In [ ]:
df_a = pd.read_parquet(FLOWS_A_OUT)
df_b = pd.read_parquet(FLOWS_B_OUT)

## Task 3: AS Resolution and IP Geolocation

In this task, we use the techniques that you have learnt in your BGP control plane assignment to map the source IPs to ASNs. Also, you will use the MaxMind GeoIP database to obtain country-level geolocation of the source IPs.

Your task is to complete the necessary functions such that `enrich_ip_src()` can add the country and ASN information as two columns in the dataframe.

In [ ]:
# students implement the following three functions. leave necessary comments to explain the code.

def build_prefix_to_asn(collector="route-views3", cache_path=None):
    """Build a prefix -> set-of-origin-ASNs map from a RouteViews RIB
    snapshot, fetched over the OSDF (Open Science Data Federation).

    This parses a full BGP RIB dump - tens of millions of routes - so
    even with bgpkit's native parser it can take several minutes and
    needs network access to OSDF. Pass cache_path to skip re-fetching
    on later runs: if that file already exists, it's loaded via
    pickle instead of rebuilding the table from scratch.
    """
    # student_code_start ------------------------------------------------
    # YOUR CODE HERE
    #
    # Hints:
    #   - If cache_path is given and Path(cache_path).exists(), load
    #     and return the pickled dict instead of rebuilding it.
    #   - Otherwise, list the RIB files for `collector` via
    #     OSDFFileSystem().ls(f"/routeviews/{collector}/bgpdata/{RIB_MONTH}/RIBS"),
    #     sort them by name, and build the download URL by prefixing
    #     the first object's name with OSDF_BASE. That takes the first RIB of
    #     the month; see the RIB_MONTH note above for why that is an
    #     approximation.
    #   - Iterate bgpkit.Parser(url=url); for each elem, update a
    #     defaultdict(set) keyed by elem.prefix with elem.origin_asns.
    #   - Convert to a plain dict, pickle it to cache_path if given,
    #     and return it.
    raise NotImplementedError
    # student_code_end --------------------------------------------------


def build_asn_trie(prefix_origins):
    """Build a pytricia trie for longest-prefix-match IP -> origin
    ASN(s) lookups from a prefix -> set-of-ASNs map."""
    # student_code_start ------------------------------------------------
    # YOUR CODE HERE
    #
    # Hints:
    #   - Create a pytricia.PyTricia().
    #   - Assign trie[prefix] = asns for every (prefix, asns) pair in
    #     prefix_origins.
    raise NotImplementedError
    # student_code_end --------------------------------------------------


def lookup_asn(trie, ip):
    """Return the origin ASN(s) covering ip as a comma-separated
    string (via longest-prefix match against trie), or None if no
    prefix in the trie covers it.
    """
    # student_code_start ------------------------------------------------
    # YOUR CODE HERE
    #
    # Hints:
    #   - trie.get(ip) performs a longest-prefix-match lookup (ip does
    #     not need to exactly equal a prefix in the trie).
    #   - Handle the "no covering prefix" case (empty/falsy result) by
    #     returning None.
    #   - Join multiple ASNs (in case of a MOAS prefix) into a single
    #     comma-separated string, sorted for consistency.
    raise NotImplementedError
    # student_code_end --------------------------------------------------

In [ ]:
def geolocate_ip(reader, ip):
    """Look up ip in an open MaxMind GeoLite2 City reader.

    Returns the country code, or None if ip isn't found (e.g.
    private/reserved ranges, which GeoLite2 doesn't cover).
    """
    import geoip2.errors

    # students implement the rest of this function, returning only the country code (or None if not found)
    # student_code_start ------------------------------------------------
    # YOUR CODE HERE
    #
    # Hints:
    #   - reader.city(ip) returns a response object with a
    #     .country.iso_code attribute (a two-letter country code).
    #   - Catch geoip2.errors.AddressNotFoundError and return None in
    #     that case.
    raise NotImplementedError
    # student_code_end --------------------------------------------------


def build_geoip_reader(source):
    """Open a MaxMind GeoLite2 database from an NRP Ceph S3 endpoint.

    Reuses the exact open/sniff approach from iter_packets() in
    the compare notebook (open via _open_binary_stream, peek for the gzip magic
    bytes rather than trusting a ".gz" suffix, wrap with
    gzip.GzipFile if present). Unlike the multi-gigabyte pcaps, a
    GeoLite2 City database is only tens of MB, so reading it fully
    into memory here (via maxminddb.MODE_FD, which just calls
    stream.read() once) is fine - no need to stream it incrementally
    the way we do for pcaps.

    Returns a geoip2.database.Reader; use it as a context manager (or
    call .close()) - the underlying stream is safely closed as soon as
    this function returns, since MODE_FD has already read all its
    bytes into the Reader by then.
    """
    with contextlib.closing(_open_binary_stream(source)) as raw:
        buffered = raw if hasattr(raw, "peek") else io.BufferedReader(raw)
        is_gzip = buffered.peek(2)[:2] == b"\x1f\x8b"
        stream = gzip.GzipFile(fileobj=buffered) if is_gzip else buffered
        return geoip2.database.Reader(stream, mode=maxminddb.MODE_FD)

In [ ]:
def enrich_ip_src(df, geoip_db_source, prefix_origins):
    """Add geolocation and AS columns for each flow's ip_src endpoint.

    Adds ip_src_country and ip_src_asn.
    """
    trie = build_asn_trie(prefix_origins)

    unique_ips = df["ip_src"].unique()
    with build_geoip_reader(geoip_db_source) as reader:
        geo_by_ip = {ip: geolocate_ip(reader, ip) for ip in unique_ips}
    asn_by_ip = {ip: lookup_asn(trie, ip) for ip in unique_ips}

    df = df.copy()
    df["ip_src_country"] = df["ip_src"].map(lambda ip: geo_by_ip[ip])
    df["ip_src_asn"] = df["ip_src"].map(asn_by_ip)
    return df

# Apply Prefix-to-AS and IP geolocation to source IPs

In [ ]:
# Build the prefix->ASN table once and reuse it for both captures -
# it's expensive enough (a full BGP RIB) that building it twice would
# be wasteful.
prefix_origins = build_prefix_to_asn(collector=COLLECTOR, cache_path=ASN_CACHE)
df_a = enrich_ip_src(df_a, GEOIP_DB, prefix_origins)
df_b = enrich_ip_src(df_b, GEOIP_DB, prefix_origins)

Print out the dataframe to check if the ip_src_country and ip_src_asn columns are correct

PCAP A:

In [ ]:
df_a.sort_values("total_bytes", ascending=False).head(20)

PCAP B:

In [ ]:
df_b.sort_values("total_bytes", ascending=False).head(20)

## Task 4: Investigate the Surge by Country

Aggregate traffic volume by `ip_src_country` for both captures.

In [ ]:
df_a.groupby('ip_src_country', dropna=False)['total_bytes'].sum().sort_values(ascending=False)

In [ ]:
df_b.groupby('ip_src_country', dropna=False)['total_bytes'].sum().sort_values(ascending=False)

### Question 4

What are the top 5 countries (sorted by traffic volume) contributing to the traffic in PCAP A and PCAP B? What is the biggest change in the ranking between A and B? Call the country with the highest traffic volume in the second capture (PCAP B) **Country<sub>top</sub>**.

YOUR ANSWER HERE

In [ ]:
# Set this to the two-letter country code you identified above as
# Country_top (the country with the highest total_bytes in df_b).
# student_code_start ------------------------------------------------
COUNTRY_TOP = None  # YOUR CODE HERE
# student_code_end --------------------------------------------------

# Left as None this does not raise - it silently matches no rows, and every
# cell below prints an empty result, which reads like a data problem.
assert COUNTRY_TOP, "Set COUNTRY_TOP from the PCAP B country ranking above."

## Task 5: Investigate the Surge by AS

Filter to Country<sub>top</sub> and aggregate traffic volume by `ip_src_asn`.

In [ ]:
df_a[df_a['ip_src_country'] == COUNTRY_TOP].groupby('ip_src_asn', dropna=False)['total_bytes'].sum().sort_values(ascending=False)

In [ ]:
df_b[df_b['ip_src_country'] == COUNTRY_TOP].groupby('ip_src_asn', dropna=False)['total_bytes'].sum().sort_values(ascending=False)

### Question 5

What are the top 5 ASes (sorted by traffic volume) within Country<sub>top</sub> in PCAP A and PCAP B? What is the biggest change in the ranking? How many origin ASes are associated with the top traffic contributor in PCAP B — what is this phenomenon called? Call this AS **AS<sub>top</sub>**.

YOUR ANSWER HERE

In [ ]:
# Set this to AS_top - the origin AS with the highest total_bytes within
# Country_top in PCAP B, as a string.
#
# That row's label may list more than one ASN: a MOAS prefix is announced by
# several origins, so the label looks like "64500,64501". Either the single
# ASN you are naming ("64500") or the whole label works - the mask in the next
# cell matches either form.
# student_code_start ------------------------------------------------
AS_TOP = None  # YOUR CODE HERE
# student_code_end --------------------------------------------------

# Guard: left as None, the mask below builds the regex "(?:^|,)None(?:,|$)",
# which matches the literal string "None" that .astype(str) produces for
# unresolved ASNs - returning real-looking rows for flows whose origin AS was
# never found.
assert AS_TOP, "Set AS_TOP from the PCAP B per-AS ranking above."

In [ ]:
# ip_src_asn can hold more than one comma-separated ASN per row (a
# MOAS prefix covers more than one origin AS), so match AS_TOP as one
# of possibly several comma-separated values rather than requiring an
# exact match.
asn_mask = df_b["ip_src_asn"].astype(str).str.contains(
    rf"(?:^|,){AS_TOP}(?:,|$)", regex=True, na=False
)
df_b[asn_mask].groupby('ip_src')['total_bytes'].sum().sort_values(ascending=False)

### Question 6

Aggregate the traffic from source IPs within AS<sub>top</sub>. Describe the distribution of traffic volume across the top 5 IPs.

YOUR ANSWER HERE

## Task 6: Investigate IP-Level Traffic

Filter to the single top source IP from Task 5 — call it **the scanner**.

In [ ]:
# Set this to the single top source IP you identified above.
# student_code_start ------------------------------------------------
SCANNER_IP = None  # YOUR CODE HERE
# student_code_end --------------------------------------------------

assert SCANNER_IP, "Set SCANNER_IP from the per-IP ranking above."

In [ ]:
scanner_flows = df_b[df_b['ip_src'] == SCANNER_IP]

print(f"flows created by the scanner: {len(scanner_flows):,}")
print(f"distinct destinations probed: {scanner_flows['ip_dst'].nunique():,}")
print(f"distinct destination ports:   {scanner_flows['port_dst'].nunique():,}")
print(f"protocols used:               {sorted(scanner_flows['proto'].unique())}")

scanner_flows

In [ ]:
df_b[df_b['ip_src'] == SCANNER_IP].groupby('ip_dst')['total_packets'].sum()

### Question 7

How many flows did the scanner create? Describe its traffic pattern — the protocol(s) it used, the destination ports it targeted, and the destinations it probed.

YOUR ANSWER HERE

## Task 7: Analyze the Scanning Behavior

See [Task 7 Guidance](Task-hilbert-map.md) for the Hilbert-curve algorithm used here.

In [ ]:
def _rot(n, x, y, rx, ry):
    if ry == 0:
        if rx == 1:
            x = n - 1 - x
            y = n - 1 - y
        x, y = y, x
    return x, y


def hilbert_d2xy(order, d):
    n = 1 << order
    x = y = 0
    t = d
    s = 1
    while s < n:
        rx = 1 & (t // 2)
        ry = 1 & (t ^ rx)
        x, y = _rot(s, x, y, rx, ry)
        x += s * rx
        y += s * ry
        t //= 4
        s *= 2
    return x, y


dst_packets = (
    df_b[df_b["ip_src"] == SCANNER_IP]
    .groupby("ip_dst")["total_packets"]
    .sum()
)

subnet_totals = {}
parsed = []
for ip, packets in dst_packets.items():
    parts = ip.split(".")
    if len(parts) != 4:
        continue
    o0, o1, o2, o3 = map(int, parts)
    subnet = f"{o0}.{o1}.0.0/16"
    subnet_totals[subnet] = subnet_totals.get(subnet, 0) + packets
    parsed.append((subnet, o2, o3, packets))

if subnet_totals:
    target_subnet = max(subnet_totals, key=subnet_totals.get)
    grid = [[0 for _ in range(256)] for _ in range(256)]

    for subnet, o2, o3, packets in parsed:
        if subnet != target_subnet:
            continue
        d = o2 * 256 + o3
        x, y = hilbert_d2xy(8, d)
        grid[y][x] += packets

    fig, ax = plt.subplots(figsize=(10, 10))
    im = ax.imshow(grid, cmap="viridis", origin="lower")

    # Gridlines every 16 px. On an order-8 curve each /24 - 256 consecutive
    # values of d - lands in exactly one aligned 16x16 cell, so these lines
    # outline the /24 blocks of the target /16.
    for i in range(0, 256, 16):
        ax.axhline(i, color='gray', linewidth=0.5, alpha=0.3)
        ax.axvline(i, color='gray', linewidth=0.5, alpha=0.3)

    # Label each /24 cell with its third octet.
    #
    # The axes are Hilbert coordinates, NOT octets: the curve reorders the
    # address space, so o2=0,o3=1 lands at (x=1, y=0) - not (0, 1). Reading a
    # block's /24 off the x/y position would be wrong; read these labels.
    for o2 in range(256):
        bx, by = hilbert_d2xy(8, o2 * 256)
        ax.text((bx // 16) * 16 + 7.5, (by // 16) * 16 + 7.5, str(o2),
                ha="center", va="center", fontsize=5, color="white", alpha=0.6)

    ax.set_xlabel("Hilbert x (curve position, not an octet)")
    ax.set_ylabel("Hilbert y (curve position, not an octet)")

    ax.set_title(f"Hilbert map of total packets in destination subnet {target_subnet}")
    fig.colorbar(im, ax=ax, label="Total packets")
    plt.show()
else:
    print("No IPv4 destination data available for Hilbert map.")

### Question 8

Which /24 blocks within that /16 did the scanner probe least?

Read the `/24` label printed inside each 16×16 cell. The x and y axes are Hilbert curve positions, **not** octets, so a block cannot be named from its axis coordinates.

YOUR ANSWER HERE

In [ ]:
data = df_b[df_b['ip_src'] == SCANNER_IP].groupby(['proto', 'port_dst'])['total_packets'].sum().reset_index()
fig, ax = plt.subplots(figsize=(10, 6))

width = 0.35
protocols = list(data["proto"].unique())
x = range(len(data["port_dst"].astype(str).unique()))
ports = sorted(data["port_dst"].unique())

for i, proto in enumerate(protocols):
    proto_data = (
        data[data["proto"] == proto]
        .set_index("port_dst")
        .reindex(ports)
        .fillna(0)
    )
    ax.bar(
        [p + (i - (len(protocols) - 1) / 2) * width for p in x],
        proto_data["total_packets"].values,
        width=width,
        label=proto,
    )

ax.set_xticks(list(x))
ax.set_xticklabels([str(p) for p in ports], rotation=45, ha="right")
ax.set_xlabel("Destination port")
ax.set_ylabel("Total packets")
ax.set_title(f"Packets from {SCANNER_IP} by destination port and protocol")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### Question 9

Using the packets-by-port-and-protocol plot above, what does the scanner's choice of destination ports and protocol reveal about its scanning pattern?

YOUR ANSWER HERE

### Question 10

Review the top flows you identified in Task 2. Why can't the scanner be detected simply by screening for the largest flows in network telescope traffic?

YOUR ANSWER HERE